In [1]:
import pandas as pd
import numpy as np

housing = pd.read_csv('datasets/housing/housing.csv')

In [2]:
from sklearn.model_selection import train_test_split

train_st, test_st = train_test_split(housing, train_size=5000, random_state=42)

train_st_labels = train_st["median_house_value"].copy()
train_st.drop("median_house_value", axis=1, inplace=True)

In [3]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.utils.validation import check_array, check_is_fitted


class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state
    def fit(self, X, y=None, sample_weight=None):
        X=check_array(X)
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        self.n_features_in_ = X.shape[1]
        return self
    def transform(self, X):
        check_is_fitted(self)
        X=check_array(X)
        assert self.n_features_in_ == X.shape[1]
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]


In [5]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector, make_column_transformer

def column_ratio(X): return X[:,[0]] / X[:,[1]]

def ratio_name(function_transformer, feature_names_in): return["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler()
    )

log_pipeline = make_pipeline( 
    SimpleImputer(strategy="median"), FunctionTransformer(np.log, feature_names_out="one-to-one"), StandardScaler()
)

cat_pipeline = make_pipeline( SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1, random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

preprocessing= ColumnTransformer(
    [
        ("bedrooms", ratio_pipeline(), ["total_bedrooms","total_rooms"]),
        ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
        ("people_per_house", ratio_pipeline(), ["population","households"]),
        ("log", log_pipeline, ["total_bedrooms","total_rooms","population", "households", "median_income"]),
        ("geo", cluster_simil, ["latitude","longitude"]),
        ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
    ],
    remainder=default_num_pipeline
)

In [6]:
from sklearn.svm import SVR

fp = Pipeline(
    [ ("preprocessing", preprocessing), ("svr", SVR(epsilon=0.2)) ]
)


In [13]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    { 'svr__kernel': ['linear'] , 'svr__C': [10., 30., 100., 1000., 50000., 65000., 89000., 120000., 240000.]  },
    #{ 'svr__kernel': ['linear'] , 'svr__C': [10., 100., 50000.] },
    #{ 'svr__kernel': ['rbf'], 'svr__C': [1.0,3.0, 100., 300., 2000.], 'svr__gamma': [0.01, 0.2, 0.5, 1., 3.0, 3.5]}
    #{ 'svr__kernel': ['rbf'], 'svr__C': [1.0, 100., 2000.], 'svr__gamma': [0.01, 0.2, 0.5, 3.5]}
]

grid_search= GridSearchCV(fp, param_grid, cv=2, scoring='neg_root_mean_squared_error')
grid_search.fit(train_st, train_st_labels)

grid_search.best_params_

{'svr__C': 65000.0, 'svr__kernel': 'linear'}

In [14]:
-grid_search.best_score_

np.float64(69254.82595275239)

In [15]:
cv_res = pd.DataFrame(grid_search.cv_results_)
model=grid_search.best_estimator_

X,y = test_st.drop("median_house_value", axis=1), test_st["median_house_value"].copy()

predictions = model.predict(X)

from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y, predictions)
rmse

247706.38671313698

In [19]:
test_st.iloc[10123]

longitude               -117.94
latitude                  33.77
housing_median_age         32.0
total_rooms               714.0
total_bedrooms            142.0
population                654.0
households                154.0
median_income            4.5052
median_house_value     170800.0
ocean_proximity       <1H OCEAN
Name: 7855, dtype: object

In [21]:
test_st.columns.drop("median_house_value")

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'ocean_proximity'],
      dtype='str')

In [22]:
test_st["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     6884
INLAND        4957
NEAR OCEAN    2035
NEAR BAY      1760
ISLAND           4
Name: count, dtype: int64

In [30]:
#datum=pd.DataFrame(np.array([121.01,40.71, 35.0, 700.,612.0, 100.0, 4.8, "NEAR OCEAN"]), columns=test_st.columns.drop("median_house_value"))
sample = {
        'longitude': -112.23,
        'latitude': 40.7, 'population':340, 'households': 118,
    'housing_median_age': 38., 'total_rooms': 808., 'total_bedrooms': 129.0, 'median_income': 8.85, 'ocean_proximity':'NEAR OCEAN'
}
datum=pd.DataFrame([sample])
y_pred = model.predict(datum)
y_pred

array([296937.7207126])

In [33]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, expon, loguniform

param_distribs = {
        'svr__kernel': ['linear', 'rbf'],
        'svr__C': loguniform(20,100_000),
        'svr__gamma': expon(scale=1.0)
}
rnd_search = RandomizedSearchCV(fp, param_distributions=param_distribs, n_iter=10, cv=3, scoring='neg_root_mean_squared_error', random_state=42)
rnd_search.fit(train_st, train_st_labels)

rnd_search.best_params_

{'svr__C': np.float64(79969.14475809233),
 'svr__gamma': np.float64(0.26497040005002437),
 'svr__kernel': 'rbf'}

In [34]:
-rnd_search.best_score_

np.float64(56695.087190633516)

In [35]:
model = rnd_search.best_estimator_
predictions = model.predict(X)

rmse = root_mean_squared_error(y, predictions)
rmse

57697.24620537945

In [38]:
y_pred=model.predict(datum)
y_pred

array([474283.46861144])

In [40]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

selector = SelectFromModel(RandomForestRegressor(random_state=42), threshold=0.005)
fp = Pipeline(
    [ ("preprocessing", preprocessing), ("feature_selection",selector), 
     ("svr", SVR(C=rnd_search.best_params_["svr__C"], gamma=rnd_search.best_params_["svr__gamma"],
                kernel='rbf'))])

rmses = -cross_val_score(fp, train_st, train_st_labels, scoring="neg_root_mean_squared_error", cv=3)
pd.Series(rmses).describe()

count        3.000000
mean     57100.464495
std       2620.065324
min      55125.457780
25%      55614.335159
50%      56103.212538
75%      58087.967853
max      60072.723167
dtype: float64

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted
from sklearn.neighbors import KNeighborsRegressor

class Smoothed_Median_Income(BaseEstimator, TranformerMixin):
    def __init__(self):
        pass
    def fit(self, X, y=None):
        
        KNeighborsRegressor.fit(X)
        pass
    def transform(self, X):
        